# torch.nn.parallel 
- 是 PyTorch 中专门用于模型并行和分布式训练的子模块。
- 它提供了多种将模型和数据分布到多个 GPU 上的策略，以适应不同的硬件配置和训练需求

## 模块结构总览

In [ ]:
torch.nn.parallel
├── DistributedDataParallel    # 分布式数据并行（推荐）
├── DataParallel               # 单机多卡数据并行（已废弃）
├── parallel_apply             # 在多个 GPU 上并行应用模块
├── replicate                  # 将模型复制到多个 GPU
├── scatter                    # 将输入数据分片到多个 GPU
├── gather                     # 从多个 GPU 收集结果
└── data_parallel              # DataParallel 的底层实现函数

## 核心组件详解

### DistributedDataParallel (DDP) — 最常用
- 定位：多机多卡分布式数据并行的标准方案。

In [ ]:
from torch.nn.parallel import DistributedDataParallel as DDP

# 基本用法
model = MyModel().cuda(local_rank)
ddp_model = DDP(
    model,
    device_ids=[local_rank],
    output_device=local_rank
)

#### 特点：
- 每个进程一个模型副本
- 数据通过 DistributedSampler 分片
- 反向传播时自动 AllReduce 同步梯度
- 支持单机多卡和多机多卡

详细示例见上一回答。

### parallel_apply — 底层并行执行
- 定位：在多个 GPU 上并行执行同一个模型的不同副本

In [ ]:
from torch.nn.parallel import parallel_apply

# 模型副本列表（每个 GPU 一个）
replicas = [model0, model1, model2, model3]  # 已在各自 GPU 上

# 输入列表（每个模型一个输入）
inputs = [input0, input1, input2, input3]

# 并行执行
outputs = parallel_apply(replicas, inputs)

# outputs = [output0, output1, output2, output3]

#### 内部实现：

In [ ]:
def parallel_apply(modules, inputs, kwargs_tup=None, devices=None):
    # 为每个 (module, input) 对创建一个线程
    # 各线程在各自 GPU 上并行执行 forward
    # 等待所有线程完成，返回结果列表

#### 使用场景：
- 模型并行（不同层在不同 GPU）
- 自定义并行策略
- DDP/DP 的底层构建块


### replicate — 模型复制
- 定位：将模型复制到多个 GPU。


In [ ]:
from torch.nn.parallel import replicate

model = MyModel().cuda(0)

# 复制到 GPU 0, 1, 2, 3
replicas = replicate(model, device_ids=[0, 1, 2, 3])

# replicas[0] 在 GPU 0，replicas[1] 在 GPU 1，...

- 注意：复制的是模型结构，参数会共享或复制（取决于设置）。

### scatter / gather — 数据分片与收集

In [ ]:
from torch.nn.parallel import scatter, gather

# ========== scatter：将数据分片到多个 GPU ==========
input_tensor = torch.randn(16, 10).cuda(0)  # 在 GPU 0

# 分片到 GPU 0, 1, 2, 3（默认按 batch 维度 dim=0）
scattered = scatter(input_tensor, target_gpus=[0, 1, 2, 3])
# scattered = [
#     tensor(4, 10) on GPU 0,
#     tensor(4, 10) on GPU 1,
#     tensor(4, 10) on GPU 2,
#     tensor(4, 10) on GPU 3
# ]

# ========== gather：从多个 GPU 收集结果 ==========
outputs = [
    torch.randn(4, 5).cuda(0),
    torch.randn(4, 5).cuda(1),
    torch.randn(4, 5).cuda(2),
    torch.randn(4, 5).cuda(3)
]

# 收集到 GPU 0
gathered = gather(outputs, target_device=0)
# gathered: tensor(16, 5) on GPU 0

### data_parallel — 底层函数
- 定位：DataParallel 的底层实现。

In [ ]:
from torch.nn.parallel import data_parallel

# 一行实现 DP 的核心逻辑
output = data_parallel(
    module=model,           # 模型
    input=input_tensor,     # 输入
    device_ids=[0, 1, 2, 3], # 使用的 GPU
    output_device=0         # 输出放到哪个 GPU
)

等价于：

In [ ]:
model = DataParallel(model, device_ids=[0, 1, 2, 3])
output = model(input_tensor)

## 各组件关系图

In [ ]:
┌─────────────────────────────────────────────────────────────┐
│                    torch.nn.parallel                         │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  ┌─────────────────────┐    ┌─────────────────────┐        │
│  │  DataParallel (DP)  │    │ DistributedDataParallel│       │
│  │  (单进程，已废弃)    │    │ (多进程，推荐)         │       │
│  └─────────────────────┘    └─────────────────────┘        │
│           ↑                            ↑                     │
│           └──────────┬─────────────────┘                     │
│                      ↓                                       │
│            ┌─────────────────┐                               │
│            │  data_parallel  │  ← 底层函数                   │
│            │  (scatter +     │                               │
│            │   parallel_apply│                               │
│            │   + gather)     │                               │
│            └─────────────────┘                               │
│                      ↑                                       │
│         ┌────────────┼────────────┐                         │
│         ↓            ↓            ↓                         │
│    ┌────────┐  ┌──────────┐  ┌────────┐                    │
│    │ scatter │  │parallel_apply│ │ gather │                    │
│    │ (数据分片)│  │ (并行执行)  │  │ (结果收集)│                    │
│    └────────┘  └──────────┘  └────────┘                    │
│                      ↑                                       │
│                 ┌────────┐                                  │
│                 │replicate│  ← 模型复制到多 GPU               │
│                 └────────┘                                  │
│                                                              │
└─────────────────────────────────────────────────────────────┘

## 使用场景选择

| 场景          | 推荐方案                           | 原因               |
| ----------- | ------------------------------ | ---------------- |
| 单机多卡（学习/调试） | DDP                            | 即使单机，DDP 也比 DP 快 |
| 多机多卡        | DDP                            | 唯一支持多机的方案        |
| 超大模型（单卡放不下） | 模型并行 + DDP                     | 层分布在不同 GPU       |
| 快速原型（不追求性能） | DP（不推荐）                        | 代码改动最小           |
| 自定义并行策略     | `parallel_apply` + `replicate` | 灵活控制             |


## 完整示例：手动组合底层 API

In [ ]:
import torch
import torch.nn as nn
from torch.nn.parallel import replicate, scatter, parallel_apply, gather

class ToyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(10, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )
    
    def forward(self, x):
        return self.net(x)

# ========== 手动实现类似 DataParallel 的逻辑 ==========

model = ToyModel().cuda(0)
device_ids = [0, 1, 2, 3]

# 1. 复制模型到多个 GPU
replicas = replicate(model, device_ids=device_ids)
# replicas[0] on GPU 0, replicas[1] on GPU 1, ...

# 2. 准备输入（在 GPU 0）
input_tensor = torch.randn(16, 10).cuda(0)

# 3. 分片数据到各 GPU
inputs = scatter(input_tensor, target_gpus=device_ids)
# inputs[0]: (4, 10) on GPU 0, inputs[1]: (4, 10) on GPU 1, ...

# 4. 并行前向传播
outputs = parallel_apply(replicas, inputs)
# outputs[0] on GPU 0, outputs[1] on GPU 1, ...

# 5. 收集结果到 GPU 0
result = gather(outputs, target_device=0)
# result: (16, 1) on GPU 0

print(f"Input shape: {input_tensor.shape}")
print(f"Output shape: {result.shape}")

## 总结

| API                       | 作用         | 使用频率    |
| ------------------------- | ---------- | ------- |
| `DistributedDataParallel` | 分布式数据并行    | ⭐⭐⭐ 最高  |
| `DataParallel`            | 单机数据并行（废弃） | ⭐ 避免使用  |
| `parallel_apply`          | 并行执行多个模型副本 | ⭐⭐ 底层开发 |
| `replicate`               | 模型复制到多 GPU | ⭐⭐ 底层开发 |
| `scatter`/`gather`        | 数据分片/收集    | ⭐⭐ 底层开发 |
| `data_parallel`           | DP 的底层函数   | ⭐ 很少直接用 |

- 核心建议：实际训练中，几乎总是用 DistributedDataParallel，其他 API 主要用于理解底层机制或开发自定义并行策略。
